# CovIntervene P0 backbone smoke

This notebook runs **smoke evidence only**: one series for each of four mechanisms and three generator seeds. It never computes the P0 continuation gate. Select a T4 GPU, choose one backbone below, and use **Runtime > Run all**. Run the notebook once for `chronos_2` and once for `timesfm_3`.

In [ ]:
BACKBONE = 'chronos_2'  # change to 'timesfm_3' for the second smoke
assert BACKBONE in {'chronos_2', 'timesfm_3'}
print('Selected frozen backbone:', BACKBONE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
print('Repository commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements/colab-base.txt')], check=True)
model_requirements = REPO / ('requirements/chronos2.txt' if BACKBONE == 'chronos_2' else 'requirements/timesfm3.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(model_requirements)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
print('Dependencies installed for', BACKBONE)

In [ ]:
import json
import torch

from covfaith.config import load_yaml, verify_config_lock

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then restart.'
config_path = REPO / 'configs/p0/covintervene_p0.yaml'
lock_path = REPO / 'configs/p0/covintervene_p0.lock.json'
config_hash = verify_config_lock(config_path, lock_path)
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(REPO / 'tests')], cwd=REPO, check=True)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Frozen config hash:', config_hash[:12])

In [ ]:
from covfaith.adapters import Chronos2Adapter, TimesFM3Adapter

config = load_yaml(config_path)
model = next(item for item in config['models'] if item['id'] == BACKBONE)
if BACKBONE == 'chronos_2':
    adapter = Chronos2Adapter.from_pretrained(
        model['checkpoint'], model['revision'], device='cuda', batch_size=128
    )
else:
    adapter = TimesFM3Adapter.from_pretrained(
        model['checkpoint'], model['revision'], device='cuda', per_core_batch_size=16
    )
print('Frozen checkpoint loaded:', model['checkpoint'], model['revision'])

In [ ]:
from covfaith.p0 import run_backbone_units

OUTPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p0_smoke_v1')
report = run_backbone_units(REPO, adapter, OUTPUT_ROOT, smoke_count=1)
assert report['scientific_gate_computed'] is False
assert report['completed_unit_count'] == 12
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Smoke artifacts:', OUTPUT_ROOT)

Send the final JSON report back to Codex. Do not run or construct the complete P0 decision yet.